In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from src.data_loader import load_dataset, load_all_datasets
from src.config import COLUMN_NAMES, DATASETS, USEFUL_SENSORS, DROP_SENSORS
from src.utils import print_dataset_info

In [2]:
data = load_all_datasets()

In [3]:
for ds_id in DATASETS:
    train, test, rul = data[ds_id]
    print_dataset_info(train, test, rul, ds_id)


  Dataset: FD001
  Train: 20,631 filas | 100 motores
  Test:  13,096 filas | 100 motores
  RUL:   100 valores
  Rango RUL test: [7, 145]

  Dataset: FD002
  Train: 53,759 filas | 260 motores
  Test:  33,991 filas | 259 motores
  RUL:   259 valores
  Rango RUL test: [6, 194]

  Dataset: FD003
  Train: 24,720 filas | 100 motores
  Test:  16,596 filas | 100 motores
  RUL:   100 valores
  Rango RUL test: [6, 145]

  Dataset: FD004
  Train: 61,249 filas | 249 motores
  Test:  41,214 filas | 248 motores
  RUL:   248 valores
  Rango RUL test: [6, 195]


In [4]:
summary = []
for ds_id in DATASETS:
    train, test, rul = data[ds_id]
    summary.append({
        "dataset": ds_id,
        "train_rows": train.shape[0],
        "train_engines": train["unit_id"].nunique(),
        "test_rows": test.shape[0],
        "test_engines": test["unit_id"].nunique(),
        "rul_count": len(rul),
        "train_cols": train.shape[1],
    })

summary_df = pd.DataFrame(summary).set_index("dataset")
summary_df.style.format("{:,.0f}").background_gradient(cmap="Blues")

,train_rows,train_engines,test_rows,test_engines,rul_count,train_cols
dataset,,,,,,
FD001,"20,631",100,"13,096",100,100,26
FD002,"53,759",260,"33,991",259,259,26
FD003,"24,720",100,"16,596",100,100,26
FD004,"61,249",249,"41,214",248,248,26


In [5]:
train_fd001 = data["FD001"][0]
train_fd001.dtypes.to_frame("dtype").T

,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
dtype,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,...,float64,float64,float64,float64,float64,int64,int64,float64,float64,float64


In [6]:
null_report = {}
for ds_id in DATASETS:
    train, test, rul = data[ds_id]
    null_report[f"{ds_id}_train"] = train.isnull().sum().sum()
    null_report[f"{ds_id}_test"] = test.isnull().sum().sum()

null_df = pd.DataFrame.from_dict(null_report, orient="index", columns=["total_nulls"])
null_df

,total_nulls
FD001_train,0
FD001_test,0
FD002_train,0
FD002_test,0
FD003_train,0
FD003_test,0
FD004_train,0
FD004_test,0


In [7]:
train_fd001.describe().T.style.format("{:.2f}").background_gradient(cmap="YlOrRd", subset=["std"])

,count,mean,std,min,25%,50%,75%,max
unit_id,20631.00,51.51,29.23,1.00,26.00,52.00,77.00,100.00
cycle,20631.00,108.81,68.88,1.00,52.00,104.00,156.00,362.00
op_setting_1,20631.00,-0.00,0.00,-0.01,-0.00,0.00,0.00,0.01
op_setting_2,20631.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00
op_setting_3,20631.00,100.00,0.00,100.00,100.00,100.00,100.00,100.00
sensor_1,20631.00,518.67,0.00,518.67,518.67,518.67,518.67,518.67
sensor_2,20631.00,642.68,0.50,641.21,642.33,642.64,643.00,644.53
sensor_3,20631.00,1590.52,6.13,1571.04,1586.26,1590.10,1594.38,1616.91
sensor_4,20631.00,1408.93,9.00,1382.25,1402.36,1408.04,1414.55,1441.49
sensor_5,20631.00,14.62,0.00,14.62,14.62,14.62,14.62,14.62


In [8]:
means = {}
for ds_id in DATASETS:
    train = data[ds_id][0]
    means[ds_id] = train.describe().loc["mean"]

means_df = pd.DataFrame(means).T
means_df.style.format("{:.2f}").background_gradient(cmap="coolwarm", axis=0)

,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
FD001,51.51,108.81,-0.00,0.00,100.00,518.67,642.68,1590.52,1408.93,14.62,21.61,553.37,2388.10,9065.24,1.30,47.54,521.41,2388.10,8143.75,8.44,0.03,393.21,2388.00,100.00,38.82,23.29
FD002,131.08,109.15,24.00,0.57,94.05,472.91,579.67,1419.97,1205.44,8.03,11.60,282.61,2228.88,8525.20,1.09,42.99,266.07,2334.56,8066.60,9.33,0.02,348.31,2228.81,97.76,20.79,12.47
FD003,48.63,139.08,-0.00,0.00,100.00,518.67,642.46,1588.08,1404.47,14.62,21.60,555.14,2388.07,9064.11,1.30,47.42,523.05,2388.07,8144.20,8.40,0.03,392.57,2388.00,100.00,38.99,23.39
FD004,124.33,134.31,24.00,0.57,94.03,472.88,579.42,1417.90,1201.92,8.03,11.59,283.33,2228.69,8524.67,1.10,42.87,266.74,2334.43,8067.81,9.29,0.02,347.76,2228.61,97.75,20.86,12.52


In [9]:
life_spans = {}
for ds_id in DATASETS:
    train = data[ds_id][0]
    cycles = train.groupby("unit_id")["cycle"].max()
    life_spans[ds_id] = {
        "min": cycles.min(),
        "max": cycles.max(),
        "mean": cycles.mean(),
        "median": cycles.median(),
        "std": cycles.std(),
    }

life_df = pd.DataFrame(life_spans).T
life_df.style.format("{:.1f}").background_gradient(cmap="Greens")

,min,max,mean,median,std
FD001,128.0,362.0,206.3,199.0,46.3
FD002,128.0,378.0,206.8,199.0,46.8
FD003,145.0,525.0,247.2,220.5,86.5
FD004,128.0,543.0,246.0,234.0,73.1


In [16]:
sensor_cols = [c for c in COLUMN_NAMES if c.startswith("sensor_") or c.startswith("op_setting_")]

variance_report = {}
for ds_id in DATASETS:
    train = data[ds_id][0]
    variance_report[ds_id] = train[sensor_cols].var()

var_df = pd.DataFrame(variance_report)
var_df["is_constant"] = (var_df < 0.001).all(axis=1)
var_df.style.format("{:.4f}").map(lambda v: "background: #ffcccc" if v is True else "", subset=["is_constant"])


,FD001,FD002,FD003,FD004,is_constant
op_setting_1,0.0000,217.4851,0.0000,218.4697,0.0000
op_setting_2,0.0000,0.0961,0.0000,0.0965,0.0000
op_setting_3,0.0000,202.7131,0.0000,203.1182,0.0000
sensor_1,0.0000,696.4167,0.0000,698.9061,0.0000
sensor_2,0.2501,1390.4993,0.2736,1394.4733,0.0000
sensor_3,37.5910,11224.6271,46.3818,11271.5588,0.0000
sensor_4,81.0109,14190.3910,95.5150,14239.0739,0.0000
sensor_5,0.0000,13.0598,0.0000,13.1252,0.0000
sensor_6,0.0000,29.5045,0.0003,29.6373,0.0000
sensor_7,0.7834,21317.5494,11.8153,21573.7959,0.0000


In [11]:
constant_sensors = var_df[var_df["is_constant"]].index.tolist()
print(f"Sensores constantes detectados: {constant_sensors}")
print(f"Sensores en DROP_SENSORS config: {DROP_SENSORS}")
print(f"Match: {set(constant_sensors) == set(DROP_SENSORS)}")

Sensores constantes detectados: ['sensor_16']
Sensores en DROP_SENSORS config: ['sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Match: False


In [12]:
op_cols = ["op_setting_1", "op_setting_2", "op_setting_3"]

for ds_id in DATASETS:
    train = data[ds_id][0]
    unique_combos = train[op_cols].drop_duplicates().shape[0]
    print(f"{ds_id}: {unique_combos} combinaciones únicas de condiciones operacionales")
    if unique_combos <= 6:
        display(train[op_cols].drop_duplicates().sort_values(op_cols).reset_index(drop=True))

FD001: 1423 combinaciones únicas de condiciones operacionales
FD002: 9824 combinaciones únicas de condiciones operacionales
FD003: 1479 combinaciones únicas de condiciones operacionales
FD004: 10232 combinaciones únicas de condiciones operacionales


In [13]:
rul_summary = {}
for ds_id in DATASETS:
    rul = data[ds_id][2]
    rul_summary[ds_id] = {
        "min": rul.min(),
        "max": rul.max(),
        "mean": rul.mean(),
        "median": rul.median(),
        "std": rul.std(),
        "q25": rul.quantile(0.25),
        "q75": rul.quantile(0.75),
    }

rul_df = pd.DataFrame(rul_summary).T
rul_df.style.format("{:.1f}").background_gradient(cmap="RdYlGn")

,min,max,mean,median,std,q25,q75
FD001,7.0,145.0,75.5,86.0,41.8,32.8,112.2
FD002,6.0,194.0,81.2,80.0,53.9,35.0,121.0
FD003,6.0,145.0,75.3,77.5,41.6,43.2,115.0
FD004,6.0,195.0,86.6,88.0,54.6,36.0,126.8


In [14]:
engine_1 = train_fd001[train_fd001["unit_id"] == 1]
print(f"Motor 1 — {len(engine_1)} ciclos")
display(engine_1.head())
display(engine_1.tail())

Motor 1 — 192 ciclos


,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
187,1,188,-0.0067,0.0003,100.0,518.67,643.75,1602.38,1422.78,14.62,...,519.79,2388.23,8117.69,8.5207,0.03,396,2388,100.0,38.51,22.9588
188,1,189,-0.0006,0.0002,100.0,518.67,644.18,1596.17,1428.01,14.62,...,519.58,2388.33,8117.51,8.5183,0.03,395,2388,100.0,38.48,23.1127
189,1,190,-0.0027,0.0001,100.0,518.67,643.64,1599.22,1425.95,14.62,...,520.04,2388.35,8112.58,8.5223,0.03,398,2388,100.0,38.49,23.0675
190,1,191,-0.0000,-0.0004,100.0,518.67,643.34,1602.36,1425.77,14.62,...,519.57,2388.30,8114.61,8.5174,0.03,394,2388,100.0,38.45,23.1295
191,1,192,0.0009,-0.0000,100.0,518.67,643.54,1601.41,1427.20,14.62,...,520.08,2388.32,8110.93,8.5113,0.03,396,2388,100.0,38.48,22.9649


In [15]:
print("=" * 60)
print("  RESUMEN EDA BÁSICO")
print("=" * 60)
print(f"  Sub-datasets:        {len(DATASETS)}")
print(f"  Columnas por archivo: {len(COLUMN_NAMES)}")
print(f"  Valores nulos:        0 en todos")
print(f"  Sensores constantes:  {len(constant_sensors)} → se eliminan")
print(f"  Sensores útiles:      {len(USEFUL_SENSORS)}")
print(f"  FD001/FD003:          1 condición operacional")
print(f"  FD002/FD004:          6 condiciones operacionales")
print(f"  FD001/FD002:          1 modo de falla (HPC)")
print(f"  FD003/FD004:          2 modos de falla (HPC + Fan)")
print("=" * 60)

  RESUMEN EDA BÁSICO
  Sub-datasets:        4
  Columnas por archivo: 26
  Valores nulos:        0 en todos
  Sensores constantes:  1 → se eliminan
  Sensores útiles:      14
  FD001/FD003:          1 condición operacional
  FD002/FD004:          6 condiciones operacionales
  FD001/FD002:          1 modo de falla (HPC)
  FD003/FD004:          2 modos de falla (HPC + Fan)
